In [1]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
from datetime import datetime
from pathlib import Path
import pandas as pd

In [4]:
PROJECT_ROOT = Path.cwd().parent   # assumes notebook lives in notebooks/, project root is one level up
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_PATH = DATA_DIR / "customer_personality.csv"

In [5]:

df = pd.read_csv(RAW_DATA_PATH)

In [6]:
df.head(5)

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,...,5,0,0,0,0,0,0,3,11,0


In [7]:
df=df.dropna()

In [8]:
# Merge rare/inconsistent marital-status categories into "Single"
df["Marital_Status"] = df["Marital_Status"].replace(
        {"Alone": "Single", "Absurd": "Single", "YOLO": "Single"}
    )

In [9]:
#Recode education levels
df["Education"] = df["Education"].replace(
    {
        "2n Cycle": "Technical School",
        "Basic": "High School",
        "Graduation": "College Graduate",
    }
)

In [10]:
#Getting age of each person
df["age"] = 2023 - df["Year_Birth"]
#drop outlier ages
df = df[df["age"] < 100]

In [11]:
#Consolidated campaign-acceptance target: any of 5 campaigns
#Response is the 6th campaign

campaign_cols = [
    "AcceptedCmp1", "AcceptedCmp2", "AcceptedCmp3", "AcceptedCmp4", "AcceptedCmp5", "Response"
]

df["Accepted_Cmp"] = (df[campaign_cols].sum(axis=1) > 0).map({True: "Yes", False: "No"})


In [13]:
 # Parse enrollment date, then compute tenure in years
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
df["duration"] = (pd.Timestamp.now() - df["Dt_Customer"]).dt.days / 365.25


In [14]:
# Combine Kidhome + Teenhome into a single "children" count
df["children"] = df["Teenhome"] + df["Kidhome"]


In [15]:
drop_cols = [
        "ID", "Year_Birth", "Kidhome", "Teenhome", "Dt_Customer", "Recency",
        "AcceptedCmp1", "AcceptedCmp2", "AcceptedCmp3", "AcceptedCmp4", "AcceptedCmp5",
        "Z_CostContact", "Z_Revenue", "Response",
    ]
df = df.drop(columns=drop_cols)

In [16]:
# Drop income outliers
df = df[df["Income"] < 200_000]

In [17]:
# Categorical dtypes for modeling convenience
for col in ["Education", "Marital_Status", "Complain", "Accepted_Cmp"]:
    df[col] = df[col].astype("category")

In [18]:
df.reset_index(drop=True)

,Education,Marital_Status,Income,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,Complain,age,Accepted_Cmp,duration,children
0,College Graduate,Single,58138.0,635,88,546,172,88,88,3,8,10,4,7,0,66,Yes,14.047912,0
1,College Graduate,Single,46344.0,11,1,6,2,1,6,2,1,1,2,5,0,69,No,12.542094,2
2,College Graduate,Together,71613.0,426,49,127,111,21,42,1,8,2,10,4,0,58,No,13.086927,0
3,College Graduate,Together,26646.0,11,4,20,10,3,5,2,2,0,4,6,0,39,No,12.613279,1
4,PhD,Married,58293.0,173,43,118,46,27,15,5,5,3,6,5,0,42,No,12.673511,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2207,College Graduate,Married,61223.0,709,43,182,42,118,247,2,9,3,4,5,0,56,No,13.275838,1
2208,PhD,Together,64014.0,406,0,30,0,0,8,7,8,2,5,7,0,77,Yes,12.284736,3
2209,College Graduate,Divorced,56981.0,908,48,217,32,12,24,1,2,3,13,6,0,42,Yes,12.657084,0
2210,Master,Together,69245.0,428,30,214,80,30,61,2,6,5,10,3,0,67,No,12.659822,1


In [19]:
purchase_cols = ["NumWebPurchases", "NumCatalogPurchases", "NumStorePurchases"]
df["HighestPurchaseSource"] = df[purchase_cols].idxmax(axis=1).map({
    "NumWebPurchases": "web",
    "NumCatalogPurchases": "catalog",
    "NumStorePurchases": "store",
})

In [20]:
df.describe()

,Income,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,age,duration,children
count,2212.000000,2212.000000,2212.000000,2212.000000,2212.000000,2212.000000,2212.000000,2212.000000,2212.000000,2212.000000,2212.000000,2212.000000,2212.000000,2212.000000,2212.000000
mean,51958.810579,305.287523,26.329566,167.029837,37.648734,27.046564,43.925859,2.324593,4.088156,2.672242,5.806510,5.321429,54.086347,13.201134,0.947559
std,21527.278844,337.322940,39.744052,224.254493,54.772033,41.090991,51.706981,1.924507,2.742187,2.927542,3.250939,2.425597,11.701599,0.554401,0.749466
min,1730.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,27.000000,12.232717,0.000000
25%,35233.500000,24.000000,2.000000,16.000000,3.000000,1.000000,9.000000,1.000000,2.000000,0.000000,3.000000,3.000000,46.000000,12.725530,0.000000
50%,51371.000000,175.500000,8.000000,68.000000,12.000000,8.000000,24.500000,2.000000,4.000000,2.000000,5.000000,6.000000,53.000000,13.207392,1.000000
75%,68487.000000,505.000000,33.000000,232.250000,50.000000,33.000000,56.000000,3.000000,6.000000,4.000000,8.000000,7.000000,64.000000,13.681040,1.000000
max,162397.000000,1493.000000,199.000000,1725.000000,259.000000,262.000000,321.000000,15.000000,27.000000,28.000000,13.000000,20.000000,83.000000,14.146475,3.000000


In [22]:
df.to_csv("../data/processed/customer_personality_cleaned.csv", index=False)